# CloudGuard — Stage 3: CNN-LSTM Temporal Drift Detection
## IEEE HiPC 2026 Submission

This notebook trains the Stage 3 CNN-LSTM model for detecting **temporal compliance drift** —
identifying when an Azure subscription's policy compliance score is trending toward a governance gap.

### What this notebook does:
1. Loads the processed dataset from Stage 1 & 2
2. Constructs 30-day sliding window sequences
3. Trains a CNN-LSTM model to predict compliance drift
4. Evaluates and saves results for the paper

### Before running:
- Runtime → Change runtime type → **GPU (T4)**
- Upload your `cloudguard_dataset.csv` file from `data/processed/` when prompted

### Outputs (download after training):
- `stage3_cnn_lstm.pt` — trained model weights
- `stage3_metrics.json` — F1, Recall, Precision, MCC, ROC-AUC
- `stage3_results.csv` — per-sequence predictions
- `stage3_training_curve.png` — loss curve for the paper

## Step 1: Install Dependencies & Check GPU

In [ ]:
# Check GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU')

# Install any missing packages
import subprocess
subprocess.run(['pip', 'install', 'scikit-learn', 'imbalanced-learn', 'matplotlib', '-q'])

## Step 2: Upload Your Dataset

In [ ]:
from google.colab import files
import io
import pandas as pd

print('Upload your cloudguard_dataset.csv file')
print('Location on your PC: C:\\Users\\chuku\\Downloads\\cloudguard\\data\\processed\\cloudguard_dataset.csv')
print()

uploaded = files.upload()

# Load the uploaded file
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'Loaded dataset: {len(df):,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print(f'Class balance:')
print(df['label'].value_counts())

## Step 3: Configuration

In [ ]:
import numpy as np
import random

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# ── Feature columns (must match config.py) ────────────────────────────────────
FEATURE_COLUMNS = [
    'pcr',                # Policy Coverage Ratio
    'dvl',                # Deployment-to-Vulnerability Lag
    'enforcement_mode',   # 1=Deny, 0=Audit
    'scope_level',        # 2=MgmtGroup, 1=Sub, 0=RG
    'policy_age_days',    # Days since policy last modified
    'resource_type_flag', # 1 if resource type has policies
    'vuln_count_30d',     # 30-day CVE count
]
TARGET_COLUMN = 'label'

# ── Sequence settings ─────────────────────────────────────────────────────────
SEQUENCE_LENGTH   = 30    # 30-day windows (T in the paper)
PREDICTION_HORIZON = 7   # Predict 7 days ahead
STEP_SIZE         = 1    # Stride between sequences

# ── Model architecture ────────────────────────────────────────────────────────
CONV_FILTERS      = 64
CONV_KERNEL_SIZE  = 3
LSTM_UNITS_1      = 128
LSTM_UNITS_2      = 64
DROPOUT_RATE      = 0.3
DENSE_UNITS       = 32

# ── Training settings ─────────────────────────────────────────────────────────
LEARNING_RATE     = 0.001
BATCH_SIZE        = 64
MAX_EPOCHS        = 100
PATIENCE          = 10    # Early stopping patience

# ── Splits ────────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Sequence length: {SEQUENCE_LENGTH} days')
print(f'Prediction horizon: {PREDICTION_HORIZON} days ahead')

## Step 4: Build Temporal Sequences

We convert the tabular dataset into time-series sequences.
Each sequence is a 30-day window of feature values, and the label
tells us whether the subscription becomes non-compliant 7 days later.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def build_sequences(df, feature_cols, target_col, seq_len, horizon, step=1):
    """
    Converts a tabular dataset into sliding window sequences for LSTM.

    Each sequence X[i] = features[i : i+seq_len]  (shape: seq_len x n_features)
    Each label   y[i] = label[i+seq_len+horizon-1] (future compliance state)

    This simulates reading a rolling 30-day compliance window and predicting
    whether a governance gap will emerge 7 days later.
    """
    # Sort by subscription_id if available, else use index order
    if 'subscription_id' in df.columns:
        df = df.sort_values(['subscription_id', df.columns.tolist()[0]]).reset_index(drop=True)

    features = df[feature_cols].values.astype(np.float32)
    labels   = df[target_col].values.astype(np.int64)

    X_sequences = []
    y_sequences = []

    total = len(features)
    for i in range(0, total - seq_len - horizon + 1, step):
        X_sequences.append(features[i : i + seq_len])
        y_sequences.append(labels[i + seq_len + horizon - 1])

    return np.array(X_sequences), np.array(y_sequences)


# ── Normalize features ────────────────────────────────────────────────────────
print('Normalizing features...')
scaler = StandardScaler()

# Fit scaler on all data, apply to features
df_features = df[FEATURE_COLUMNS].copy()
df_features_scaled = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=FEATURE_COLUMNS
)
df_scaled = df.copy()
df_scaled[FEATURE_COLUMNS] = df_features_scaled

print(f'Feature means after scaling: {df_features_scaled.mean().round(3).tolist()}')
print(f'Feature stds after scaling:  {df_features_scaled.std().round(3).tolist()}')

# ── Build sequences ───────────────────────────────────────────────────────────
print(f'\nBuilding {SEQUENCE_LENGTH}-day sequences...')
X, y = build_sequences(
    df_scaled, FEATURE_COLUMNS, TARGET_COLUMN,
    SEQUENCE_LENGTH, PREDICTION_HORIZON, STEP_SIZE
)

print(f'Total sequences: {len(X):,}')
print(f'Sequence shape:  {X.shape}  (n_sequences x seq_len x n_features)')
print(f'Label shape:     {y.shape}')
print(f'Class balance:   {(y==0).sum():,} compliant / {(y==1).sum():,} non-compliant')
print(f'Non-compliant %: {(y==1).mean()*100:.1f}%')

# ── Train/val/test split ──────────────────────────────────────────────────────
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=SEED, stratify=y
)
val_size_adj = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size_adj, random_state=SEED, stratify=y_temp
)

print(f'\nSplits:')
print(f'  Train: {len(X_train):,} sequences ({(y_train==1).sum():,} non-compliant)')
print(f'  Val:   {len(X_val):,} sequences ({(y_val==1).sum():,} non-compliant)')
print(f'  Test:  {len(X_test):,} sequences ({(y_test==1).sum():,} non-compliant)')

## Step 5: PyTorch Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from imblearn.over_sampling import SMOTE

class ComplianceSequenceDataset(Dataset):
    """PyTorch Dataset for compliance sequences."""

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)   # shape: (n, seq_len, n_features)
        self.y = torch.LongTensor(y)    # shape: (n,)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ── Apply SMOTE to training sequences ─────────────────────────────────────────
# SMOTE works on 2D data, so we flatten sequences, apply SMOTE, then reshape
print('Applying SMOTE to training sequences...')
n_train, seq_len, n_features = X_train.shape
X_train_flat = X_train.reshape(n_train, seq_len * n_features)

smote = SMOTE(k_neighbors=5, random_state=SEED)
X_train_sm_flat, y_train_sm = smote.fit_resample(X_train_flat, y_train)
X_train_sm = X_train_sm_flat.reshape(-1, seq_len, n_features)

print(f'Before SMOTE: {(y_train==1).sum():,} non-compliant / {len(y_train):,} total')
print(f'After SMOTE:  {(y_train_sm==1).sum():,} non-compliant / {len(y_train_sm):,} total')

# ── Create DataLoaders ────────────────────────────────────────────────────────
train_dataset = ComplianceSequenceDataset(X_train_sm, y_train_sm)
val_dataset   = ComplianceSequenceDataset(X_val, y_val)
test_dataset  = ComplianceSequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'\nDataLoaders ready:')
print(f'  Train batches: {len(train_loader)}')
print(f'  Val batches:   {len(val_loader)}')
print(f'  Test batches:  {len(test_loader)}')

## Step 6: CNN-LSTM Model Architecture

The model follows the paper's Section IV-C description:
- **1D CNN**: Extracts local temporal patterns from the sequence
- **LSTM (2 layers)**: Captures long-range temporal dependencies
- **Dense head**: Maps to binary compliance prediction

In [ ]:
import torch.nn as nn

class CloudGuardCNNLSTM(nn.Module):
    """
    CloudGuard Stage 3: CNN-LSTM for temporal compliance drift detection.

    Architecture (Section IV-C of the paper):
        Input:  (batch, seq_len=30, n_features=7)

        CNN block:
            Conv1D(64 filters, kernel=3) → ReLU → MaxPool1D(2)
            → Extracts local temporal patterns (e.g., rapid PCR drops)

        LSTM block:
            LSTM(128 units, 2 layers, dropout=0.3)
            → Captures long-range dependencies across the 30-day window

        Dense head:
            Dropout(0.3) → Linear(128→32) → ReLU → Linear(32→2)
            → Binary classification: compliant vs non-compliant
    """

    def __init__(self, n_features, seq_len,
                 conv_filters=64, conv_kernel=3,
                 lstm_units_1=128, lstm_units_2=64,
                 dropout=0.3, dense_units=32):
        super(CloudGuardCNNLSTM, self).__init__()

        self.n_features = n_features
        self.seq_len    = seq_len

        # ── CNN Block ──────────────────────────────────────────────────────────
        # Conv1D expects input: (batch, channels, length)
        # We treat features as channels, time steps as length
        self.conv_block = nn.Sequential(
            nn.Conv1d(
                in_channels=n_features,
                out_channels=conv_filters,
                kernel_size=conv_kernel,
                padding=conv_kernel // 2  # 'same' padding
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),  # Halves the sequence length
            nn.Dropout(dropout),
        )

        # Calculate the sequence length after CNN+pooling
        self.cnn_output_len = seq_len // 2

        # ── LSTM Block ────────────────────────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size=conv_filters,     # CNN output features
            hidden_size=lstm_units_1,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=False
        )

        # ── Dense Classification Head ─────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_units_1, dense_units),
            nn.ReLU(),
            nn.Linear(dense_units, 2)    # 2 classes: compliant, non-compliant
        )

    def forward(self, x):
        """
        Forward pass.

        Args:
            x: Input tensor of shape (batch, seq_len, n_features)

        Returns:
            Logits of shape (batch, 2)
        """
        # CNN expects (batch, channels, length) — swap seq and feature dims
        x_cnn = x.permute(0, 2, 1)               # (batch, n_features, seq_len)
        x_cnn = self.conv_block(x_cnn)            # (batch, conv_filters, seq_len/2)

        # LSTM expects (batch, seq_len, features)
        x_lstm = x_cnn.permute(0, 2, 1)          # (batch, seq_len/2, conv_filters)
        x_lstm, _ = self.lstm(x_lstm)             # (batch, seq_len/2, lstm_units)

        # Use the last LSTM output (summary of entire sequence)
        x_last = x_lstm[:, -1, :]                 # (batch, lstm_units)

        # Classification
        logits = self.classifier(x_last)          # (batch, 2)
        return logits


# ── Instantiate model ─────────────────────────────────────────────────────────
model = CloudGuardCNNLSTM(
    n_features=len(FEATURE_COLUMNS),
    seq_len=SEQUENCE_LENGTH,
    conv_filters=CONV_FILTERS,
    conv_kernel=CONV_KERNEL_SIZE,
    lstm_units_1=LSTM_UNITS_1,
    lstm_units_2=LSTM_UNITS_2,
    dropout=DROPOUT_RATE,
    dense_units=DENSE_UNITS
).to(DEVICE)

print('Model architecture:')
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## Step 7: Training

In [ ]:
from sklearn.metrics import f1_score

# ── Loss function with class weighting ────────────────────────────────────────
# Weight the non-compliant class higher since it's our primary detection target
class_counts = np.bincount(y_train_sm)
class_weights = torch.FloatTensor(
    [1.0, class_counts[0] / class_counts[1]]
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ── Learning rate scheduler ───────────────────────────────────────────────────
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

print(f'Class weights: compliant={class_weights[0]:.3f}, non-compliant={class_weights[1]:.3f}')
print(f'Starting training for up to {MAX_EPOCHS} epochs (early stop after {PATIENCE} no-improve)...')
print()

# ── Training loop ─────────────────────────────────────────────────────────────
train_losses, val_losses, val_f1s = [], [], []
best_val_f1   = 0.0
best_epoch    = 0
patience_count = 0

for epoch in range(MAX_EPOCHS):

    # ── Training phase ────────────────────────────────────────────────────────
    model.train()
    epoch_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)

    # ── Validation phase ──────────────────────────────────────────────────────
    model.eval()
    epoch_val_loss = 0.0
    val_preds, val_true = [], []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            epoch_val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_true.extend(y_batch.cpu().numpy())

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_f1 = f1_score(val_true, val_preds, pos_label=1, zero_division=0)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_f1s.append(val_f1)

    scheduler.step(val_f1)

    # ── Early stopping ────────────────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch + 1
        patience_count = 0
        # Save best model
        torch.save(model.state_dict(), 'stage3_cnn_lstm_best.pt')
    else:
        patience_count += 1

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{MAX_EPOCHS} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f} | '
              f'Val F1: {val_f1:.4f} | '
              f'Best F1: {best_val_f1:.4f} (epoch {best_epoch})')

    if patience_count >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}. Best val F1: {best_val_f1:.4f} at epoch {best_epoch}')
        break

print(f'\nTraining complete. Best model saved from epoch {best_epoch}.')

## Step 8: Evaluate on Test Set

In [ ]:
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, matthews_corrcoef, classification_report
)
import json

# Load the best saved model
model.load_state_dict(torch.load('stage3_cnn_lstm_best.pt', map_location=DEVICE))
model.eval()

all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)

        logits = model(X_batch)
        probs  = torch.softmax(logits, dim=1)[:, 1]  # Probability of non-compliant
        preds  = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y_batch.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)
all_probs = np.array(all_probs)

# ── Calculate metrics ─────────────────────────────────────────────────────────
f1        = f1_score(all_true, all_preds, pos_label=1, zero_division=0)
recall    = recall_score(all_true, all_preds, pos_label=1, zero_division=0)
precision = precision_score(all_true, all_preds, pos_label=1, zero_division=0)
mcc       = matthews_corrcoef(all_true, all_preds)
roc_auc   = roc_auc_score(all_true, all_probs)

metrics = {
    'model':        'CNN-LSTM',
    'f1':           round(float(f1), 4),
    'recall':       round(float(recall), 4),
    'precision':    round(float(precision), 4),
    'mcc':          round(float(mcc), 4),
    'roc_auc':      round(float(roc_auc), 4),
    'best_epoch':   best_epoch,
    'sequence_len': SEQUENCE_LENGTH,
    'horizon':      PREDICTION_HORIZON,
}

print('=' * 55)
print('  Stage 3 -- CNN-LSTM Test Set Results')
print('=' * 55)
print(f'  F1 Score:  {f1:.4f}  <- primary metric')
print(f'  Recall:    {recall:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  MCC:       {mcc:.4f}')
print(f'  ROC-AUC:   {roc_auc:.4f}')
print('=' * 55)
print()
print('Detailed Classification Report:')
print(classification_report(
    all_true, all_preds,
    target_names=['Compliant (0)', 'Non-Compliant (1)'],
    zero_division=0
))

# Save metrics
with open('stage3_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: stage3_metrics.json')

## Step 9: Save Training Curve & Results

In [ ]:
import matplotlib.pyplot as plt

# ── Training curve ────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(train_losses) + 1)

ax1.plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, val_losses,   'r-', label='Val Loss',   linewidth=2)
ax1.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Stage 3 CNN-LSTM — Training & Validation Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_range, val_f1s, 'g-', linewidth=2, label='Val F1')
ax2.axvline(x=best_epoch, color='r', linestyle='--', alpha=0.7, label=f'Best F1={best_val_f1:.4f}')
ax2.axhline(y=best_val_f1, color='r', linestyle=':', alpha=0.5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score (Non-Compliant Class)')
ax2.set_title('Stage 3 CNN-LSTM — Validation F1 Score')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('stage3_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: stage3_training_curve.png')

# ── Save results CSV ──────────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'true_label':    all_true,
    'predicted':     all_preds,
    'prob_noncompliant': all_probs,
})
results_df.to_csv('stage3_results.csv', index=False)
print('Saved: stage3_results.csv')

# ── Save final model ──────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'metrics':          metrics,
    'scaler_mean':      scaler.mean_.tolist(),
    'scaler_std':       scaler.scale_.tolist(),
    'feature_columns':  FEATURE_COLUMNS,
    'sequence_length':  SEQUENCE_LENGTH,
}, 'stage3_cnn_lstm.pt')
print('Saved: stage3_cnn_lstm.pt')

## Step 10: Download All Output Files

In [ ]:
from google.colab import files

print('Downloading output files...')
print('Save these to your cloudguard/outputs/ folder on your PC')
print()

for filename in ['stage3_cnn_lstm.pt', 'stage3_metrics.json',
                 'stage3_results.csv', 'stage3_training_curve.png']:
    try:
        files.download(filename)
        print(f'Downloaded: {filename}')
    except Exception as e:
        print(f'Could not download {filename}: {e}')

print()
print('After downloading, place files in:')
print('  stage3_cnn_lstm.pt        -> cloudguard/outputs/models/')
print('  stage3_metrics.json       -> cloudguard/outputs/results/')
print('  stage3_results.csv        -> cloudguard/outputs/results/')
print('  stage3_training_curve.png -> cloudguard/outputs/figures/')

print()
print('Stage 3 Summary:')
print(f'  F1 Score:  {metrics["f1"]}')
print(f'  Recall:    {metrics["recall"]}')
print(f'  Precision: {metrics["precision"]}')
print(f'  MCC:       {metrics["mcc"]}')
print(f'  ROC-AUC:   {metrics["roc_auc"]}')
print(f'  Best epoch: {metrics["best_epoch"]}')

## Step 11: MLP Baseline (Ablation — Table VIII)

This trains a simple MLP without temporal modeling to isolate
the contribution of the CNN-LSTM sequence architecture.
The difference between MLP and CNN-LSTM F1 scores is the
evidence for temporal modeling value in your paper.

In [ ]:
class MLPBaseline(nn.Module):
    """Non-temporal MLP baseline for ablation study (Table VIII)."""

    def __init__(self, n_features, seq_len, hidden=128, dropout=0.3):
        super(MLPBaseline, self).__init__()
        input_size = n_features * seq_len
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


mlp = MLPBaseline(len(FEATURE_COLUMNS), SEQUENCE_LENGTH).to(DEVICE)
mlp_optimizer = torch.optim.Adam(mlp.parameters(), lr=LEARNING_RATE)
mlp_criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Training MLP baseline (non-temporal)...')

mlp_best_f1 = 0.0
mlp_patience = 0

for epoch in range(MAX_EPOCHS):
    mlp.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        mlp_optimizer.zero_grad()
        loss = mlp_criterion(mlp(X_batch), y_batch)
        loss.backward()
        mlp_optimizer.step()

    mlp.eval()
    preds, true = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            p = torch.argmax(mlp(X_batch.to(DEVICE)), dim=1)
            preds.extend(p.cpu().numpy())
            true.extend(y_batch.numpy())

    vf1 = f1_score(true, preds, pos_label=1, zero_division=0)
    if vf1 > mlp_best_f1:
        mlp_best_f1 = vf1
        mlp_patience = 0
        torch.save(mlp.state_dict(), 'stage3_mlp_baseline_best.pt')
    else:
        mlp_patience += 1
    if mlp_patience >= PATIENCE:
        print(f'MLP early stop at epoch {epoch+1}, best val F1: {mlp_best_f1:.4f}')
        break

# Evaluate MLP on test set
mlp.load_state_dict(torch.load('stage3_mlp_baseline_best.pt', map_location=DEVICE))
mlp.eval()
mlp_preds, mlp_true, mlp_probs = [], [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = mlp(X_batch.to(DEVICE))
        probs  = torch.softmax(logits, dim=1)[:, 1]
        preds  = torch.argmax(logits, dim=1)
        mlp_preds.extend(preds.cpu().numpy())
        mlp_true.extend(y_batch.numpy())
        mlp_probs.extend(probs.cpu().numpy())

mlp_f1  = f1_score(mlp_true, mlp_preds, pos_label=1, zero_division=0)
mlp_auc = roc_auc_score(mlp_true, mlp_probs)

print()
print('Ablation Results (Table VIII):')
print(f'  CNN-LSTM F1:  {f1:.4f}   ROC-AUC: {roc_auc:.4f}')
print(f'  MLP F1:       {mlp_f1:.4f}   ROC-AUC: {mlp_auc:.4f}')
print(f'  F1 improvement from temporal modeling: {(f1 - mlp_f1)*100:.2f} percentage points')